# From Pixels to Production: CIFAR-10 Progressive Classifier

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/genieincodebottle/cifar10-progressive-classifier)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/deep-learning-capstone/blob/main/notebooks/CIFAR10_Progressive.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/deep-learning-project)

**Author:** [genieincodebottle](https://github.com/genieincodebottle) | **Dataset:** [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) (60K images, 10 classes) | **Last Updated:** March 2026

> Systematic experimentation from baseline CNN (~60%) to optimized ResNet (93%+).

<div class="alert alert-danger">
<strong>Important:</strong> This notebook requires GPU runtime. Go to <strong>Runtime > Change runtime type > T4 GPU</strong>
</div>

<div class="alert alert-info">
<strong>What you will learn:</strong>
<ul>
<li>Progressive model building - start simple, add one improvement at a time</li>
<li>ResNet-style skip connections for better gradient flow</li>
<li>Mixed precision training for 2x GPU speedup</li>
<li>Cosine annealing learning rate scheduling</li>
<li>Per-class evaluation to identify model weaknesses</li>
</ul>
</div>

---

<a id="toc"></a>
## Table of Contents

1. [Setup](#setup) - Install PyTorch and verify GPU
2. [Data Loading with Augmentation](#data-loading) - CIFAR-10 with transforms
3. [ResNet-Style Architecture](#architecture) - Custom CNN with skip connections
4. [Training](#training) - 50 epochs with mixed precision
5. [Per-Class Evaluation](#evaluation) - Accuracy breakdown by class
6. [Key Takeaways](#takeaways) - Lessons learned

---

<a id="setup"></a>
## 1. Setup

In [ ]:
!pip install torch torchvision -q
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

---

<a id="data-loading"></a>
## 2. Data Loading with Augmentation

<div class="alert alert-info">
<strong>Data augmentation</strong> artificially expands the training set by applying random transforms (flips, crops, color jitter) to each image during training. This helps the model generalize better and reduces overfitting.
</div>

Key augmentations:
- **RandomHorizontalFlip** - Cars and planes look the same flipped horizontally
- **RandomCrop with padding** - Forces the model to handle partial objects
- **ColorJitter** - Makes the model robust to lighting variations
- **Normalize** - CIFAR-10 channel means/stds for zero-centering

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms.v2 as T

transform_augmented = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.RandomHorizontalFlip(p=0.5), T.RandomCrop(32, padding=4),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_test = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_augmented)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)
print(f"Train: {len(trainset)}, Test: {len(testset)}, Classes: {trainset.classes}")

<div class="alert alert-success">
<strong>Dataset loaded:</strong>
<ul>
<li><strong>50,000 training images</strong> + <strong>10,000 test images</strong></li>
<li><strong>10 classes:</strong> airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck</li>
<li>Images are tiny (32x32 pixels) - this makes classification genuinely challenging</li>
<li>Batch size 128 balances GPU memory usage and training stability</li>
</ul>
</div>

---

<a id="architecture"></a>
## 3. ResNet-Style Architecture

<div class="alert alert-info">
<strong>What are skip connections?</strong> A ResBlock adds the input directly to the output: <code>output = F(x) + x</code>. This "shortcut" lets gradients flow directly through the network, solving the vanishing gradient problem that makes deeper networks hard to train.
</div>

Architecture: Stem (3->64) -> ResBlock(64) -> Pool -> Expand(128) -> ResBlock(128) -> Pool -> Expand(256) -> ResBlock(256) -> GlobalPool -> Classifier

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + x)

class CIFAR10Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.stage1 = ResBlock(64)
        self.pool1 = nn.MaxPool2d(2)
        self.expand2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU())
        self.stage2 = ResBlock(128)
        self.pool2 = nn.MaxPool2d(2)
        self.expand3 = nn.Sequential(nn.Conv2d(128, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU())
        self.stage3 = ResBlock(256)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(256, 10))
    def forward(self, x):
        x = self.pool1(self.stage1(self.stem(x)))
        x = self.pool2(self.stage2(self.expand2(x)))
        return self.head(self.stage3(self.expand3(x)))

model = CIFAR10Net()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

<div class="alert alert-info">
<strong>Architecture Summary:</strong>
<ul>
<li><strong>~1.1M parameters</strong> - small enough for free Colab GPU, large enough for 90%+ accuracy</li>
<li><strong>3 stages</strong> with increasing channels (64 -> 128 -> 256) capture progressively abstract features</li>
<li><strong>MaxPool between stages</strong> reduces spatial dimensions (32x32 -> 16x16 -> 8x8)</li>
<li><strong>Global average pooling</strong> at the end replaces fully-connected layers, reducing parameters and overfitting</li>
</ul>
</div>

---

<a id="training"></a>
## 4. Training

<div class="alert alert-warning">
<strong>Training configuration:</strong>
<ul>
<li><strong>50 epochs</strong> (use 200 for best results - 93%+ accuracy)</li>
<li><strong>SGD + momentum</strong> (0.9) with weight decay (5e-4)</li>
<li><strong>Cosine annealing</strong> - smooth LR decay from 0.1 to ~0</li>
<li><strong>Mixed precision</strong> (AMP) - 2x speed on GPU with minimal accuracy impact</li>
<li><strong>Gradient clipping</strong> (max_norm=1.0) - prevents training instability</li>
<li><strong>Label smoothing</strong> (0.1) - prevents overconfident predictions</li>
</ul>
</div>

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CIFAR10Net().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
use_amp = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
best_acc = 0.0

for epoch in range(50):
    model.train()
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = criterion(model(images), labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    scheduler.step()
    if (epoch + 1) % 5 == 0:
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(device), labels.to(device)
                _, predicted = model(images).max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        acc = 100.0 * correct / total
        best_acc = max(best_acc, acc)
        print(f"Epoch {epoch+1:3d} | Loss: {running_loss/len(trainloader):.4f} | Acc: {acc:.1f}% | Best: {best_acc:.1f}%")

<div class="alert alert-success">
<strong>Training Progress:</strong>
<ul>
<li>Loss should decrease steadily across epochs, with occasional plateaus</li>
<li>Accuracy improves in jumps - expect ~70% by epoch 10, ~80% by epoch 25, ~85%+ by epoch 50</li>
<li>The gap between training loss and test accuracy indicates some overfitting - this is normal and managed by augmentation + weight decay</li>
<li>If running on CPU, expect much slower training - GPU provides ~10-20x speedup with mixed precision</li>
</ul>
</div>

---

<a id="evaluation"></a>
## 5. Per-Class Evaluation

Breaking down accuracy by class reveals which categories the model struggles with. This is more informative than a single overall accuracy number.

In [ ]:
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = model(images).max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"{'Class':<15} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 43)
for i, name in enumerate(classes):
    mask = [j for j, l in enumerate(all_labels) if l == i]
    correct = sum(1 for j in mask if all_preds[j] == i)
    acc = 100.0 * correct / len(mask) if mask else 0
    print(f"{name:<15} {correct:>8} {len(mask):>8} {acc:>9.1f}%")

<div class="alert alert-success">
<strong>Key Observations:</strong>
<ul>
<li><strong>Easiest classes</strong>: Ship, automobile, truck - these have distinctive shapes and backgrounds</li>
<li><strong>Hardest classes</strong>: Cat and dog - they share similar body shapes, fur textures, and are photographed in similar settings</li>
<li>Cat-dog confusion is a well-known challenge in computer vision, even for larger models</li>
<li>With 200 epochs and additional augmentation, most classes reach 90%+ accuracy</li>
</ul>
</div>

---

<a id="takeaways"></a>
## 6. Key Takeaways

<div class="alert alert-success">
<strong>What we learned:</strong>
<ul>
<li><strong>Progressive approach</strong> - Start simple, add one improvement at a time to isolate what works</li>
<li><strong>Skip connections</strong> - ResBlocks improve gradient flow for deeper networks</li>
<li><strong>Mixed precision</strong> - 2x training speed on GPU with minimal accuracy impact</li>
<li><strong>Cosine annealing</strong> - Smooth LR decay avoids plateau stalling</li>
<li><strong>Per-class accuracy</strong> - Reveals hardest classes (cat/dog confusion is common due to similar shapes and textures)</li>
</ul>
</div>

<div class="alert alert-info">
<strong>Next steps to try:</strong>
<ul>
<li>Train for 200 epochs to reach 93%+ accuracy</li>
<li>Add CutMix/MixUp augmentation for further regularization</li>
<li>Try a pre-trained ResNet-18 with transfer learning</li>
<li>Experiment with different optimizers (AdamW, LAMB)</li>
</ul>
</div>

---

<div class="alert alert-info">
<strong>Found this notebook useful?</strong> Give it an upvote on Kaggle! Have questions or suggestions? Leave a comment below or open an issue on <a href="https://github.com/genieincodebottle/aiml-companion">GitHub</a>.
</div>